In [24]:
import json
import time
import requests
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

In [25]:
LONDON_LAT, LONDON_LON = 51.5074, -0.1278

def get_weather(past_steps=96, forecast_steps=96):
    """15-min weather for London. 96 steps = 24 hours.

    Returns DataFrame with: time, temperature, wind_speed, humidity,
    precipitation, cloud_cover, visibility, apparent_temperature.
    """
    variables = "temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,wind_speed_10m,cloud_cover,visibility"
    resp = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": LONDON_LAT, "longitude": LONDON_LON,
        "minutely_15": variables,
        "past_minutely_15": past_steps,
        "forecast_minutely_15": forecast_steps,
        "timezone": "Europe/London",
    })
    resp.raise_for_status()
    m = resp.json()["minutely_15"]
    return pd.DataFrame({
        "time": pd.to_datetime(m["time"]).tz_localize("Europe/London"),
        "temperature": m["temperature_2m"],
        "apparent_temperature": m["apparent_temperature"],
        "humidity": m["relative_humidity_2m"],
        "precipitation": m["precipitation"],
        "wind_speed": m["wind_speed_10m"],
        "cloud_cover": m["cloud_cover"],
        "visibility": m["visibility"],
    })

In [26]:
df_weather = get_weather()
print(f"{len(df_weather)} readings, {df_weather.time.min()} -> {df_weather.time.max()}")
df_weather.tail(5)

192 readings, 2026-02-28 01:15:00+00:00 -> 2026-03-02 01:00:00+00:00


,time,temperature,apparent_temperature,humidity,precipitation,wind_speed,cloud_cover,visibility
187,2026-03-02 00:00:00+00:00,10.5,7.5,79,0.0,15.1,100,28100.0
188,2026-03-02 00:15:00+00:00,10.4,7.5,78,0.0,14.8,100,27860.0
189,2026-03-02 00:30:00+00:00,10.4,7.5,77,0.0,14.4,100,27620.0
190,2026-03-02 00:45:00+00:00,10.4,7.4,76,0.0,14.4,100,27380.0
191,2026-03-02 01:00:00+00:00,10.4,7.4,75,0.0,14.0,100,27140.0


In [27]:
df_today = df_weather[(df_weather['time'] >= pd.to_datetime('2026-02-28 12:00+00:00')) & (df_weather['time'] < pd.to_datetime('2026-03-01 12:00+00:00'))]

In [28]:
df_today

,time,temperature,apparent_temperature,humidity,precipitation,wind_speed,cloud_cover,visibility
43,2026-02-28 12:00:00+00:00,9.6,6.1,69,0.0,15.1,91,19120.0
44,2026-02-28 12:15:00+00:00,9.8,6.2,68,0.0,14.8,92,19480.0
45,2026-02-28 12:30:00+00:00,9.8,6.3,67,0.0,14.8,94,19840.0
46,2026-02-28 12:45:00+00:00,9.9,6.3,67,0.0,14.8,96,20200.0
47,2026-02-28 13:00:00+00:00,9.9,6.3,66,0.0,14.8,97,20560.0
...,...,...,...,...,...,...,...,...
134,2026-03-01 10:45:00+00:00,10.9,8.1,84,0.0,15.8,100,14860.0
135,2026-03-01 11:00:00+00:00,10.9,8.1,84,0.0,16.2,100,14600.0
136,2026-03-01 11:15:00+00:00,11.0,8.1,84,0.0,16.9,100,14740.0
137,2026-03-01 11:30:00+00:00,11.0,8.1,84,0.0,17.6,100,14880.0


In [29]:
df_today['f'] = (df_today['temperature'] * (9/5)) + 32

/tmp/ipykernel_1046420/600084343.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_today['f'] = (df_today['temperature'] * (9/5)) + 32


In [30]:
df_today['TH'] = (df_today['f'] * df_today['humidity']) / 100

/tmp/ipykernel_1046420/4096116722.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_today['TH'] = (df_today['f'] * df_today['humidity']) / 100


In [31]:
df_today['TH'].sum()

np.float64(3228.4883999999997)

In [19]:
df_settle = df_weather[df_weather['time'] == pd.to_datetime('2026-02-28 12:00+00:00')]

In [21]:
df_settle['f'] = (df_settle['temperature'] * (9/5)) + 32

/tmp/ipykernel_1046420/3320272199.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_settle['f'] = (df_settle['temperature'] * (9/5)) + 32


In [23]:
df_settle['f'] * df_settle['humidity']

46    3400.32
dtype: float64

WX_SUM
15 min refresh
Spread fixed size 9
For every net position, skew theo offset by 1